# LegalMitra: LoRA Adapter Merge & 4-bit GGUF Export

**Clean, Zero-Error Architecture:**
1. **Eliminates `torchao` Conflict:** Uninstalls `torchao` (or upgrades/bypasses it) so PEFT standard LoRA dispatch executes cleanly without version mismatches.
2. **Native CPU Quantizer:** Compiled via CMake cleanly without missing CUDA driver errors.
3. **Outputs `legalmitra-8b-q4_k_m.gguf` (~4.6 GB)** ready for offline local CPU inference.

**Hardware Setting:** Kaggle Free GPU (T4 x2 or P100) | **Internet:** Enabled

## Step 1: Install Dependencies, Remove Incompatible torchao & Build llama.cpp

In [ ]:
# 1. Remove buggy pre-installed torchao from Kaggle environment
!pip uninstall -y torchao

# 2. Install required packages
!pip install -q "transformers>=4.44.0" "peft>=0.12.0" "accelerate>=0.33.0" sentencepiece huggingface_hub gguf

# 3. Clone llama.cpp and build CPU conversion & quantization binaries with CMake
!rm -rf /kaggle/working/llama.cpp
!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git /kaggle/working/llama.cpp
!cmake -B /kaggle/working/llama.cpp/build -S /kaggle/working/llama.cpp -DGGML_CUDA=OFF
!cmake --build /kaggle/working/llama.cpp/build --config Release -j4 --target llama-quantize llama-cli
!pip install -q -r /kaggle/working/llama.cpp/requirements.txt

print("SUCCESS: Environment configured and llama.cpp built cleanly.")

## Step 2: Locate the Uploaded LoRA Adapter Dataset

In [ ]:
import os
import glob

# Locate the adapter folder automatically across Kaggle input and working directories
adapter_candidates = glob.glob('/kaggle/input/**/adapter_model.safetensors', recursive=True)

if adapter_candidates:
    adapter_dir = os.path.dirname(adapter_candidates[0])
    print(f"Found LegalMitra LoRA adapter at: {adapter_dir}")
else:
    adapter_dir = '/kaggle/working/legalmitra_llama3_lora_final'
    if not os.path.exists(adapter_dir):
        raise FileNotFoundError(
            "LoRA adapter not found! Please attach your 'legalmitra-lora-adapter' Kaggle dataset "
            "via '+ Add Input' on the right panel."
        )
    print(f"Using local session adapter at: {adapter_dir}")

print("Adapter directory files:", os.listdir(adapter_dir))

with open('/kaggle/working/adapter_path.txt', 'w') as f:
    f.write(adapter_dir)

## Step 3: Merge LoRA Adapter with Base Model (Isolated PyTorch Process)

In [ ]:
merge_script = '''
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

# Monkey-patch peft.import_utils to report torchao as unavailable
import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

with open("/kaggle/working/adapter_path.txt", "r") as f:
    adapter_dir = f.read().strip()

base_model_name = "unsloth/Meta-Llama-3.1-8B-Instruct"
merged_output_dir = "/kaggle/working/legalmitra_merged_fp16"

print(f"Loading base model in FP16 on GPU: {base_model_name}...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
)

print(f"Applying LegalMitra LoRA adapter from: {adapter_dir}...")
model = PeftModel.from_pretrained(base_model, adapter_dir)

print("Merging LoRA weights directly into the base weights...")
merged_model = model.merge_and_unload()

print(f"Saving merged FP16 model to: {merged_output_dir}...")
merged_model.save_pretrained(merged_output_dir, safe_serialization=True)
tokenizer.save_pretrained(merged_output_dir)
print("SUCCESS: Model successfully merged into full FP16 weights.")
'''

with open('/kaggle/working/merge_runner.py', 'w') as f:
    f.write(merge_script)

!python3 /kaggle/working/merge_runner.py

## Step 4: Convert Merged Model to GGUF & Quantize to 4-bit (q4_k_m)

In [ ]:
import os

fp16_gguf_path = "/kaggle/working/legalmitra-8b-fp16.gguf"
q4_gguf_path = "/kaggle/working/legalmitra-8b-q4_k_m.gguf"

print("Step 4A: Converting merged FP16 weights to GGUF format...")
!python3 /kaggle/working/llama.cpp/convert_hf_to_gguf.py \
    /kaggle/working/legalmitra_merged_fp16 \
    --outfile {fp16_gguf_path} \
    --outtype f16

print("Step 4B: Quantizing FP16 GGUF to q4_k_m (4-bit Medium)...")
quantize_bin = "/kaggle/working/llama.cpp/build/bin/llama-quantize"

!{quantize_bin} {fp16_gguf_path} {q4_gguf_path} q4_k_m

# Clean up large intermediate 16GB FP16 files to free disk space
if os.path.exists(fp16_gguf_path):
    os.remove(fp16_gguf_path)
!rm -rf /kaggle/working/legalmitra_merged_fp16
if os.path.exists('/kaggle/working/merge_runner.py'):
    os.remove('/kaggle/working/merge_runner.py')

q4_size_gb = os.path.getsize(q4_gguf_path) / (1024 ** 3)
print(f"\n====================================================================")
print(f"SUCCESS: Standalone GGUF model created: {q4_gguf_path}")
print(f"Final 4-bit Quantized Size: {q4_size_gb:.2f} GB")
print(f"====================================================================")
print("You can now download 'legalmitra-8b-q4_k_m.gguf' from the Output tab on the right panel!")

## Step 5: Quick Inference Test (Sanity Check)

In [ ]:
test_prompt = "<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\nWhat happens in case of minor theft under the Bharatiya Nyaya Sanhita, 2023?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

cli_bin = "/kaggle/working/llama.cpp/build/bin/llama-cli"

!{cli_bin} \
    -m /kaggle/working/legalmitra-8b-q4_k_m.gguf \
    -p "{test_prompt}" \
    -n 128 \
    --temp 0.2